# Capstone companion --- Chapter 1: What Is an Agent? The Governed Loop

Chapter~1 defines an agent not as a model but as a controlled decision system that wraps a model: a governed loop that runs *observe -> validate -> plan -> authorize -> act -> verify -> log*. The model proposes; the system disposes. This companion reads that definition on the capstone banking complaint agent, where a language model is surrounded by a typed toolset, a stack of admission gates and a hash-chained audit log. The agent is the governance around the model, not the model alone.

The capstone is assembled by `build_complaint_harness`, which returns a `GovernanceHarness` wrapping a `ComplaintAgent` and a `ToolRegistry`. The harness loads the language model and the trained geometric store, so it is GPU-resident and is not constructed here. The classes that constitute it are imported and read structurally, which is sufficient to see how the governed loop is realized.

In [ ]:
import inspect
from forgeloop.agents.capstone.complaint_agent import (
    ComplaintAgent,
    build_complaint_harness,
)
from forgeloop.agents.governance.harness import GovernanceHarness
from forgeloop.agents.tools import GovernedToolExecutor, ToolRegistry
from forgeloop.agents.audit.logger import AuditLogger

print('agent   :', ComplaintAgent.__mro__[1].__name__, '-> ComplaintAgent')
print('builder :', inspect.signature(build_complaint_harness))

## The model is wrapped, not run directly

An agent in the Chapter~1 sense never lets the model act on the world directly. The model's role is confined to *proposing* an action; the harness decides whether that action is admitted. `ComplaintAgent` implements this by overriding a single method, `propose_action`, which returns a typed `Action` (a tool call, an escalation or a finish) given the current state. Everything between the proposal and the effect is the governance layer.

In [ ]:
src = inspect.getsource(ComplaintAgent.propose_action)
# The agent returns one of three typed actions; it never touches a tool itself.
for kind in ('ToolCall', 'Escalate', 'Finish'):
    print(f'{kind:10s} proposed in propose_action:', kind in src)

## Plan: the governed workflow is a fixed order of typed actions

The *plan* stage of the loop is explicit in the capstone. The complaint agent walks a fixed sequence of workflow nodes, each a typed tool call, and short-circuits to an escalation on any prior failure or regulatory flag. The node order below is the plan the model's proposals are checked against; a proposed transition that the trained store judges implausible for this order is refused at the plausibility gate.

In [ ]:
from forgeloop.agents.capstone.complaint_agent import _WORKFLOW_NODES, _TOOL_NODE_MAP

print('workflow node order (the plan):')
for i, node in enumerate(_WORKFLOW_NODES):
    print(f'  {i}. {node}')
print('tool -> node aliases:', _TOOL_NODE_MAP)

## Validate and authorize: the gate stack around the model

Between the plan and the act, the harness runs an ordered stack of admission gates through the `GovernedToolExecutor`. Each gate is a check that returns *allow*, *deny* or *escalate*; the first non-allow decision stops the call before the tool body runs. The capstone stacks a syntax gate (schema validation), a policy gate (PII, prompt injection, prohibited advice) and a trained geometric plausibility gate. The gate stack is the *validate* and *authorize* stages made concrete.

In [ ]:
from forgeloop.agents.governance import GateDecision, SyntaxGate

# The three admissible gate decisions -- the vocabulary of authorization.
print('gate decisions:', [d.value for d in GateDecision])

# GovernedToolExecutor runs its gate list in order and stops at the first
# non-allow decision (see execute()): validate/authorize before act.
exec_src = inspect.getsource(GovernedToolExecutor.execute)
print('stops on DENY     :', 'GateDecision.DENY' in exec_src)
print('stops on ESCALATE :', 'GateDecision.ESCALATE' in exec_src)

The gate stack the capstone assembles is built in `build_complaint_harness`: a `SyntaxGate`, the policy engine rendered as a gate, and the GMS plausibility gate. Reading the builder's source shows the exact composition without loading any model.

In [ ]:
build_src = inspect.getsource(build_complaint_harness)
gate_line = next(l.strip() for l in build_src.splitlines() if l.strip().startswith('gates ='))
print('gate stack :', gate_line)
policy_line = next(l.strip() for l in build_src.splitlines() if 'pii_policy' in l)
print('policies   :', 'pii + semantic prompt-injection + prohibited-advice')

## Verify and log: the effect is checked and recorded

After a tool acts, the loop *verifies* the result and *logs* the step. The complaint agent verifies structurally before output leaves the system: before the model writes the customer-facing draft it asserts that every claim carries an evidence pointer, and it honors an escalation demanded by the draft verifier. The `GovernanceHarness` then writes a hash-chained `AuditEvent` for every step, so the whole trajectory can be replayed and its integrity checked.

In [ ]:
# The audit log is a tamper-evident hash chain: log() seals each event and
# verify() checks the chain. This is the 'log' stage of the governed loop.
logger = AuditLogger()
print('AuditLogger.log     :', inspect.signature(AuditLogger.log))
print('AuditLogger.verify  :', inspect.signature(AuditLogger.verify))
print('empty chain verifies:', logger.verify())
print('events so far       :', len(logger.events))

## The naive loop and the governed loop, stage by stage

The chapter also fixes the *minimal agent loop* and contrasts the naive loop, observe -> think -> act, with the governed loop above. The naive loop is a single call site: the model returns an action and the runtime invokes it immediately. The governed loop separates the agent that *proposes* an action from the executor that decides whether it may *run*, interposing validation and authorization before the action and verification and logging after it. Each governed stage maps to a concrete capstone symbol.

In [ ]:
# Governed-loop stages mapped to the capstone symbol that realizes each one.
stages = [
    ('observe',   'AgentState.tool_results (prior observations)'),
    ('validate',  'SyntaxGate.check -> schema validation'),
    ('plan',      'ComplaintAgent.propose_action -> Action'),
    ('authorize', 'PolicyEngine.as_gate + GMSPlausibilityGate'),
    ('act',       'GovernedToolExecutor.execute -> tool.fn(**args)'),
    ('verify',    'GMS draft verifier + assert_all_claims_have_evidence'),
    ('log',       'GovernanceHarness.run -> AuditEvent per step'),
]
for stage, symbol in stages:
    print(f'{stage:10s} <- {symbol}')

### Observing the plan without running any tool

The plan stage can be inspected in isolation because `propose_action` reads only `state`. Constructing an `AgentState` with no tool results yet places the agent at step~0, where it proposes the classifier. This is the first half of one iteration of the loop --- the proposal --- with no execution and no model call.

In [ ]:
from forgeloop.agents.core.state import AgentState
from forgeloop.agents.core.task import TaskSpec

agent = ComplaintAgent()
task = TaskSpec(
    goal='handle one banking complaint',
    inputs={'message': 'I was charged a $35 overdraft fee I did not authorize.'},
)
state0 = AgentState(task=task)          # step 0, no observations yet
action0 = agent.propose_action(state0)
print('kind      :', action0.kind)
print('tool_name :', action0.tool_name)
print('arguments :', action0.arguments)

### Validation rejects a malformed call at the boundary

The `SyntaxGate` is the validate stage in its cheapest form. It looks the tool up in the registry and validates the proposed arguments against the input schema; a missing or wrong-typed field is a `DENY` before any execution. The demonstration below builds a registry, wraps it in an executor with only the syntax gate, and offers it a malformed call.

In [ ]:
from pathlib import Path
from forgeloop.agents.tools.executor import SyntaxGate
from forgeloop.agents.capstone import banking_tools

root = next((c for c in (Path('.'), Path('..'), Path('../code'), Path('code')) if (c / 'data').exists()), Path('.'))
reg = ToolRegistry()
banking_tools.register_all(reg, policies_dir=root / 'data' / 'policies')
executor = GovernedToolExecutor(reg, gates=[SyntaxGate()])

from forgeloop.agents.core.action import ToolCall
bad = ToolCall(tool_name='classify_complaint', arguments={})  # missing 'message'
result = executor.execute(bad, state=None)
print('success     :', result.success)
print('error       :', result.error)
print('gate verdict :', [(g.gate_name, g.decision.value) for g in result.gate_results])

## Reference: how the harness is built at run time

The cell below constructs the full capstone harness. It loads the language model and the trained geometric store onto the GPU, so it is shown as reference and is left unexecuted in this teaching notebook. It is the single call that ties the model, the toolset, the gate stack and the audit log into the controlled decision system of Chapter~1.

In [ ]:
# Reference only -- loads Qwen + the GMS store (GPU). Do not run in a
# lightweight environment.
#
# from pathlib import Path
# harness, registry = build_complaint_harness(
#     policies_dir=Path('data') / 'policies',
# )
# assert isinstance(harness, GovernanceHarness)
# assert isinstance(registry, ToolRegistry)
# # harness.run(task) drives observe -> validate -> plan -> authorize ->
# # act -> verify -> log for one complaint, returning a Trajectory and
# # sealing an audit event per step.
print('build_complaint_harness returns:', 'tuple[GovernanceHarness, ToolRegistry]')

This is the capstone's realization of Chapter~1. The language model contributes only proposals through `propose_action`; the `GovernedToolExecutor` supplies *validate* and *authorize* as an ordered gate stack; the tools supply *act*; the evidence and draft checks supply *verify*; and the `GovernanceHarness` supplies *log* as a hash-chained audit trail. The agent is the governed loop around the model, which is the definition Chapter~1 sets out. Chapter~16 assembles the five tools, the full gate stack and the audit log into the running complaint agent and evaluates it end to end.